# From Reviews to Revenue — Results Walkthrough

This notebook reads the saved outputs of Stages 1–4 (`data/processed/`) and shows the key results with the code that produces them. It does not re-run the NLP models.

- All money is in **BRL (R$)**.
- All statistics are **associations, not causal effects**.
- Exposure figures are **scenario-based estimates, not observed revenue loss**.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data' / 'processed'
S4 = DATA / 'stage4'
pd.set_option('display.float_format', '{:,.4f}'.format)

## Stage 1 — Star schema and text coverage

In [2]:
tables = ['fact_orders','fact_order_items','fact_order_reviews','dim_customers',
          'dim_products','dim_sellers','dim_payments','dim_geolocation','nlp_reviews']
pd.DataFrame({t: [len(pd.read_parquet(DATA / f'{t}.parquet'))] for t in tables}, index=['rows']).T

,rows
fact_orders,99441
fact_order_items,112650
fact_order_reviews,99224
dim_customers,99441
dim_products,32951
dim_sellers,3095
dim_payments,99440
dim_geolocation,19015
nlp_reviews,40977


In [3]:
reviews = pd.read_parquet(DATA / 'fact_order_reviews.parquet')
has_text = reviews['review_comment_message'].notna()
print(f'Reviews with text: {has_text.sum():,} of {len(reviews):,} ({has_text.mean():.1%})')
reviews['review_score'].value_counts().sort_index().to_frame('reviews')

Reviews with text: 40,977 of 99,224 (41.3%)


,reviews
review_score,
1,11424
2,3151
3,8179
4,19142
5,57328


## Stage 3 — NLP results

In [4]:
nlp = json.loads((DATA / 'nlp_summary_metrics.json').read_text(encoding='utf-8'))
t = nlp['transformer_sentiment']
print('Model:', t['model'])
print('Agreement with star rating:', t['agreement_rate_pct'], '%  | lexicon baseline:',
      nlp['baseline_lexicon']['agreement_rate_pct'], '%')
pd.Series(t['distribution']).to_frame('reviews')

Model: cardiffnlp/twitter-xlm-roberta-base-sentiment
Agreement with star rating: 69.88 %  | lexicon baseline: 65.76 %


,reviews
Positive,21493
Negative,9896
Neutral,9588


In [5]:
pd.read_csv(DATA / 'dim_topics.csv')[['topic_id','business_topic_name','review_count','representative_terms']]

,topic_id,business_topic_name,review_count,representative_terms
0,0,Positive Shopping Experience,5864,"produto, recomendo, prazo, entrega, loja, gost..."
1,1,Order Receipt / Delivery Problems,5193,"recebi, comprei, entrega, veio, produto, ainda..."
2,2,Product Quality Praise,4056,"produto, qualidade, entrega, bom, recomendo, ó..."
3,3,Early Delivery Satisfaction,6462,"prazo, antes, produto, antes prazo, chegou, en..."
4,4,General Approval / Short Praise,4005,"bom, recomendo, tudo, excelente, ótimo, ok, go..."
5,5,Mixed Delivery + Product Feedback,5365,"entrega, produto, bom, recomendo, prazo, otimo..."
6,6,Product Expectation vs Reality,4759,"produto, veio, comprei, qualidade, recebi, che..."
7,7,Delivery Status / Pending Orders,5273,"produto, recebi, ainda, recebi produto, entreg..."


In [6]:
asp = pd.DataFrame(nlp['aspect_summary']).T[['total_mentions','negative','neutral','positive']].astype(int)
asp['neg_pct'] = (asp['negative'] / asp['total_mentions'] * 100).round(1)
asp.sort_values('neg_pct', ascending=False)

,total_mentions,negative,neutral,positive,neg_pct
Wrong/Missing Item,1817,1136,605,76,62.5000
Packaging,874,485,236,153,55.5000
Price/Value,1790,695,386,709,38.8000
Customer Service,4918,1889,907,2122,38.4000
Description Mismatch,1853,633,649,571,34.2000
Product Quality,18450,5326,3894,9230,28.9000
Delivery/Logistics,18181,4028,4640,9513,22.2000
Uncategorized,11497,1695,2571,7231,14.7000


## Stage 4 — Statistics (one row per text review on a delivered order)

In [7]:
s4 = json.loads((S4 / 'stage4_metrics.json').read_text(encoding='utf-8'))
st = s4['statistical_analysis']
print(st['unit_of_analysis'], '| N =', st['n_reviews_analysed'])
pd.DataFrame(st['correlations']).T

one row per text review on a delivered order; price and freight_value are order-level totals | N = 39100


,rho,p_value,n
Delivery Delay vs Review Score,-0.2172,0.0000,"39,100.0000"
Freight Value vs Review Score,-0.1193,0.0000,"39,100.0000"
Price vs Review Score,-0.0551,0.0000,"39,100.0000"
Complaint Rate vs Avg Rating,-0.9813,0.0000,37.0000
Negative Sentiment Rate vs Avg Rating,-0.8388,0.0000,37.0000


In [8]:
o = st['ontime_vs_late']
pd.DataFrame({'mean_score': [o['ontime_mean'], o['late_mean']],
              'neg_review_rate': [o['ontime_neg_review_rate'], o['late_neg_review_rate']],
              'n': [o['ontime_n'], o['late_n']]}, index=['on time', 'late'])

,mean_score,neg_review_rate,n
on time,3.9800,0.1842,34824
late,2.1200,0.6719,4276


In [9]:
lr = st['logistic_regression']
print(lr['disclaimer'], '| pseudo R2 =', lr['pseudo_r2'])
pd.DataFrame({'coef': lr['coefficients'], 'odds_ratio': lr['odds_ratios'], 'p_value': lr['p_values']})

Observational data only. Coefficients are associations, not causal effects. | pseudo R2 = 0.0791


,coef,odds_ratio,p_value
const,-0.8797,0.4149,0.0000
delay_days,0.0629,1.0649,0.0000
price,0.0000,1.0000,0.6691
freight_value,0.0119,1.0120,0.0000


## Stage 4 — Scenario-based exposure and prioritization

> Scenario-based estimate, not observed historical revenue loss.

In [10]:
e = s4['commercial_exposure']
print(f"Order value tied to negative NLP sentiment: R$ {e['affected_order_value_BRL']:,.2f} "
      f"({e['affected_orders']:,} orders)")
pd.DataFrame(e['sensitivity_table'])

Order value tied to negative NLP sentiment: R$ 1,761,899.57 (9,529 orders)


,impact_pct,exposure_BRL
0,1.0000,"17,619.0000"
1,2.0000,"35,237.9900"
2,5.0000,"88,094.9800"
3,10.0000,"176,189.9600"
4,15.0000,"264,284.9400"
5,20.0000,"352,379.9100"


In [11]:
prio = pd.read_csv(S4 / 'business_prioritization.csv')
prio.groupby('segment').agg(categories=('category','count'),
                            reviews=('review_count','sum'),
                            item_value_BRL=('total_value','sum')).round(0)

,categories,reviews,item_value_BRL
segment,,,
HIGH Risk + HIGH Exposure,10,"20,135.0000","3,289,844.0000"
HIGH Risk + LOW Exposure,8,"1,694.0000","307,687.0000"
LOW Risk + HIGH Exposure,8,"13,880.0000","2,330,590.0000"
LOW Risk + LOW Exposure,11,"3,742.0000","614,259.0000"


In [12]:
prio[prio['segment'] == 'HIGH Risk + HIGH Exposure'].sort_values('neg_sentiment_rate', ascending=False)

,category,review_count,neg_review_rate,neg_sentiment_rate,avg_order_value,total_value,segment
0,office_furniture,608.0000,0.3684,0.3651,281.5348,"169,287.2100",HIGH Risk + HIGH Exposure
3,unknown,628.0000,0.3519,0.2994,165.3613,"99,757.0100",HIGH Risk + HIGH Exposure
6,computers_accessories,"2,733.0000",0.3055,0.2755,173.0110,"467,559.6600",HIGH Risk + HIGH Exposure
7,telephony,"1,853.0000",0.2644,0.2725,103.5490,"188,783.3800",HIGH Risk + HIGH Exposure
8,furniture_decor,"2,608.0000",0.2991,0.2634,157.0397,"390,244.8000",HIGH Risk + HIGH Exposure
9,baby,"1,075.0000",0.3107,0.2623,193.1900,"200,966.2700",HIGH Risk + HIGH Exposure
13,toys,"1,448.0000",0.2452,0.2438,160.9958,"230,343.8600",HIGH Risk + HIGH Exposure
15,watches_gifts,"2,491.0000",0.2605,0.2405,241.9348,"600,018.6200",HIGH Risk + HIGH Exposure
16,housewares,"2,368.0000",0.2470,0.2394,148.5201,"344,224.6400",HIGH Risk + HIGH Exposure
17,bed_bath_table,"4,323.0000",0.2760,0.2355,142.3668,"598,658.1500",HIGH Risk + HIGH Exposure


## SQL layer — the same numbers from DuckDB

In [13]:
from src.analytics.validate_sql import connect
sql = (ROOT / 'sql' / '09_complaint_breakdown.sql').read_text(encoding='utf-8')
connect().execute(sql).df()

,topic,negative_reviews,share_of_negative_pct,avg_review_score
0,Delivery Status / Pending Orders,2954,29.9000,1.4820
1,Order Receipt / Delivery Problems,2832,28.6000,1.9640
2,Product Expectation vs Reality,2797,28.3000,2.1660
3,Mixed Delivery + Product Feedback,933,9.4000,2.5990
4,Early Delivery Satisfaction,215,2.2000,3.1350
5,Positive Shopping Experience,93,0.9000,3.3760
6,Product Quality Praise,41,0.4000,3.4630
7,General Approval / Short Praise,31,0.3000,4.3230
